In [2]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

In [3]:
import pandas as pd

earthquakes = pd.read_csv("data/parsed.csv")

earthquakes.head()

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,status,time,title,tsunami,type,types,tz,updated,url,parsed_place
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.008693,NaN,85.0,",ci37389218,",1.35,ml,...,automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...,California
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.020030,NaN,79.0,",ci37389202,",1.29,ml,...,automatic,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...,California
2,NaN,4.4,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.021370,28.0,21.0,",ci37389194,",3.42,ml,...,automatic,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.0,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...,California
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.026180,NaN,39.0,",ci37389186,",0.44,ml,...,automatic,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...,California
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.077990,NaN,192.0,",nc73096941,",2.16,md,...,automatic,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.0,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...,California


In [4]:
earthquakes.columns

Index(['alert', 'cdi', 'code', 'detail', 'dmin', 'felt', 'gap', 'ids', 'mag',
       'magType', 'mmi', 'net', 'nst', 'place', 'rms', 'sig', 'sources',
       'status', 'time', 'title', 'tsunami', 'type', 'types', 'tz', 'updated',
       'url', 'parsed_place'],
      dtype='object')

In [5]:
japan_mb = earthquakes[
    earthquakes["place"].str.contains("Japan", case=False, na=False)
    & earthquakes["magType"].eq("mb")
]

japan_95th_percentile = japan_mb["mag"].quantile(0.95)

print("95th percentile:", japan_95th_percentile)

95th percentile: 4.9


In [6]:
indonesia = earthquakes[
    earthquakes["place"].str.contains("Indonesia", case=False, na=False)
]

indonesia_tsunami_percentage = indonesia["tsunami"].eq(1).mean() * 100

print("Percentage of earthquakes coupled with tsunamis:",
      indonesia_tsunami_percentage, "%")

Percentage of earthquakes coupled with tsunamis: 23.12925170068027 %


In [7]:
nevada = earthquakes[
    earthquakes["place"].str.contains("Nevada", case=False, na=False)
]

nevada.describe()

,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.000000,677.000000,15.000000,677.000000,677.000000,1.00,677.000000,677.000000,677.000000,6.770000e+02,677.0,677.0,6.770000e+02
mean,2.440000,0.166982,2.400000,154.029527,0.491728,2.84,12.608567,0.151909,10.688331,1.538314e+12,0.0,-480.0,1.538402e+12
std,0.501142,0.166400,4.626013,68.769713,0.689560,NaN,9.890620,0.084742,19.252727,5.954070e+08,0.0,0.0,6.000267e+08
min,2.000000,0.001000,1.000000,29.140000,-0.500000,2.84,3.000000,0.000500,0.000000,1.537247e+12,0.0,-480.0,1.537307e+12
25%,2.000000,0.054000,1.000000,97.670000,-0.100000,2.84,6.000000,0.106900,0.000000,1.537854e+12,0.0,-480.0,1.537928e+12
50%,2.200000,0.113000,1.000000,149.550000,0.400000,2.84,9.000000,0.146300,2.000000,1.538280e+12,0.0,-480.0,1.538428e+12
75%,2.900000,0.234000,1.000000,200.470000,0.900000,2.84,16.000000,0.186700,12.000000,1.538821e+12,0.0,-480.0,1.538878e+12
max,3.300000,1.414000,19.000000,355.910000,2.900000,2.84,61.000000,0.863400,129.000000,1.539461e+12,0.0,-480.0,1.539483e+12


In [9]:
ring_of_fire_pattern = (
    r"Bolivia|Chile|Ecuador|Peru|Costa Rica|Guatemala|"
    r"(?<!New )Mexico|Japan|Philippines|Indonesia|New Zealand|"
    r"Antarctic|Canada|Fiji|Alaska|Washington|California|"
    r"Russia|Taiwan|Tonga|Kermadec Islands"
)

earthquakes["ring_of_fire"] = earthquakes["place"].str.contains(
    ring_of_fire_pattern,
    case=False,
    na=False,
    regex=True
)

earthquakes[["place", "ring_of_fire"]].head()

,place,ring_of_fire
0,"9km NE of Aguanga, CA",False
1,"9km NE of Aguanga, CA",False
2,"8km NE of Aguanga, CA",False
3,"9km NE of Aguanga, CA",False
4,"10km NW of Avenal, CA",False


In [11]:
ring_of_fire_count = earthquakes["ring_of_fire"].sum()
outside_ring_count = (~earthquakes["ring_of_fire"]).sum()

print("Earthquakes in the Ring of Fire:", ring_of_fire_count)
print("Earthquakes outside the Ring of Fire:", outside_ring_count)

Earthquakes in the Ring of Fire: 4426
Earthquakes outside the Ring of Fire: 4906


In [12]:
ring_of_fire_tsunami_count = earthquakes.loc[
    earthquakes["ring_of_fire"], "tsunami"
].eq(1).sum()

print("Tsunami count along the Ring of Fire:",
      ring_of_fire_tsunami_count)

Tsunami count along the Ring of Fire: 43
